# Multi-Layer Yield Curve Network Evolution Analysis

This notebook extends the network evolution framework to multi-layer networks using yield curve zero rates data.
Each layer represents a different maturity term (6M, 1Y, 2Y, 5Y, etc.), with nodes as bond issuers.
Inter-layer edges connect the same issuer across maturity terms, creating a temporal yield curve structure.

## Key Concepts
- **Intra-layer edges**: Correlations between issuers within the same term
- **Inter-layer edges**: Direct connections linking the same issuer across terms
- **Temporal evolution**: Rolling window networks over time for each layer + aggregate multi-layer structure
- **Multi-layer metrics**: Degree centrality, clustering, betweenness accounting for all layers

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import networkx as nx
from pathlib import Path
import duckdb
from tqdm import tqdm
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import seaborn as sns
from plotnine import *

# TGraph imports
from tgraphportfolio.analysis.measures import compute_measure, available_measures
from tgraphportfolio.analysis.network import build_corr_nx, pivot_to_wide
from tgraphportfolio.analysis.evolution import (
    EvolutionConfig, CommunityMethod, compute_community_metrics
)

print(f"Imports successful at {datetime.now()}")

## Chapter A: Data Loading and Exploration

Load zero-coupon yield rates from DuckDB and understand the multi-layer structure.

In [ ]:
# Connect to DuckDB
db_path = r'D:\data\duckdb\ycs_data.duckdb'
conn = duckdb.connect(db_path, read_only=True)

# Load zero_rates data
df_raw = conn.execute(
    "SELECT * FROM zero_rates ORDER BY date, source"
).pl()

print(f"Loaded {df_raw.height:,} rows, {df_raw.width} columns")
print(f"\nColumns: {df_raw.columns}")
print(f"\nData types:")
print(df_raw.schema)
print(f"\nFirst 10 rows:")
print(df_raw.head(10))

In [ ]:
df_raw = df_raw.unpivot(on=['Y000p5', 'Y001p0', 'Y001p5', 'Y002p0', 'Y002p5', 'Y003p0', 
                   'Y003p5', 'Y004p0', 'Y004p5', 'Y005p0', 'Y006p0', 'Y007p0', 
                   'Y008p0', 'Y009p0', 'Y010p0', 'Y011p0', 'Y012p0', 'Y013p0', 
                   'Y014p0', 'Y015p0', 'Y016p0', 'Y017p0', 'Y018p0', 'Y019p0', 
                   'Y020p0', 'Y025p0', 'Y030p0'], index=['date','source'], variable_name="term", value_name="rate")

In [ ]:
# Explore the structure
n_sources = df_raw.select('source').n_unique()
n_terms = df_raw.select('term').n_unique()
date_range = df_raw.select(['date']).describe()

print(f"Sources (nodes): {n_sources}")
print(f"\nTerms (layers): {n_terms}")
print(f"\nDate range:")
print(date_range)

# Show available terms
terms_sorted = df_raw.select('term').unique().sort('term')
print(f"\nTerms (sorted): {terms_sorted['term'].to_list()}")

# Show available sources
sources = df_raw.select('source').unique().sort('source')
print(f"\nSources (unique issuers): {sources['source'].to_list()}")

In [ ]:
# Check for missing data
print("Missing values per column:")
for col in df_raw.columns:
    n_nulls = df_raw.select(pl.col(col).null_count())[0, 0]
    pct = 100.0 * n_nulls / df_raw.height
    print(f"  {col}: {n_nulls:,} ({pct:.2f}%)")

# Time series completeness
print("\nCompleteness by date:")
completeness = df_raw.group_by('date').agg(
    pl.len().alias('count'),
    (pl.col('rate').null_count()).alias('nulls')
).with_columns(
    pct_complete=(100.0 * (pl.col('count') - pl.col('nulls')) / pl.col('count'))
).sort('date')
print(completeness)

In [ ]:
coverage = (
    df_raw
    .with_columns(
        pl.col("date").str.to_date()
    )
    .group_by("source")
    .agg(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
        (pl.col("date").max() - pl.col("date").min())
        .dt.total_days()
        .alias("coverage_days")
    )
    .sort("coverage_days")
)

In [ ]:
(
    ggplot(coverage, aes(x="min_date", y="source"))
    + geom_errorbarh(
        aes(xmin="min_date", xmax="max_date"),
        height=0.4
    )
    + labs(
        title="Coverage by Source",
        x="Date",
        y="Source"
    )
    + theme_minimal() + theme(figure_size=(10,4))
)

## Chapter B: Single-Layer Network Construction

Build intra-layer networks for each term independently, using rolling windows.

In [ ]:
# Prepare data: remove nulls, sort
df_clean = df_raw.drop_nulls(subset=['date', 'source', 'term', 'rate']).sort(['date', 'source', 'term'])

print(f"After removing nulls: {df_clean.height:,} rows")

# Extract unique dates and terms for rolling window setup
dates_unique = sorted(df_clean.select('date').unique()['date'].to_list())
terms_unique = sorted(df_clean.select('term').unique()['term'].to_list())
sources_unique = sorted(df_clean.select('source').unique()['source'].to_list())

print(f"\nUnique dates: {len(dates_unique)}")
print(f"Unique terms: {len(terms_unique)}")
print(f"Unique sources: {len(sources_unique)}")

print(f"\nDate range: {dates_unique[0]} to {dates_unique[-1]}")

In [ ]:
# Function to build a single-layer network for a specific term over a date window
def build_term_network(
    df: pl.DataFrame,
    term: str,
    date_start,
    date_end,
    measure: str = 'spearman_correlation',
    independent_threshold: float = 0.33
) -> nx.Graph:
    """
    Build a single-layer network for a specific term in a date window.
    Nodes are sources (issuers), edges are correlations of rates over time.
    """
    # Filter by term and date range
    df_term = df.filter(
        (pl.col('term') == term) &
        (pl.col('date') >= date_start) &
        (pl.col('date') <= date_end)
    )
    
    if df_term.height < 3:
        return nx.Graph()  # Empty graph if insufficient data
    
    # Pivot to wide: rows=dates, cols=sources
    df_wide = pivot_to_wide(
        df_term,
        date_column='date',
        name_column='source',
        value_column='rate'
    )
    
    nodes = [c for c in df_wide.columns if c != 'date']
    if len(nodes) < 2:
        return nx.Graph()  # Need at least 2 nodes
    
    # Compute correlation measure
    measure_df = compute_measure(
        measure,
        df_wide.select(nodes),
        nodes,
        progress=None
    )
    
    # Build network
    G = build_corr_nx(measure_df, independent_threshold=independent_threshold)
    
    return G

print("Function defined: build_term_network")

In [ ]:
# Test on a single term with a rolling window
window_size = 125  # 125 trading days
step_size = 21    # Step by 21 days

# Use first term for testing
test_term = terms_unique[0]
print(f"Testing with term: {test_term}")

# Create rolling windows
windows = []
for i in range(0, len(dates_unique) - window_size + 1, step_size):
    date_start = dates_unique[i]
    date_end = dates_unique[i + window_size - 1]
    windows.append((date_start, date_end))

print(f"\nNumber of windows: {len(windows)}")
print(f"First window: {windows[0][0]} to {windows[0][1]}")
print(f"Last window: {windows[-1][0]} to {windows[-1][1]}")

In [ ]:
# Compute intra-layer networks for each term and window
layer_networks = {}  # {term: [G0, G1, ..., Gt]}
layer_metadata = {}  # {term: [(date_start, date_end, |V|, |E|), ...]}

for term in tqdm(terms_unique, desc='Terms'):
    networks = []
    metadata = []
    
    for date_start, date_end in tqdm(windows, desc=f'  {term}', leave=False):
        G = build_term_network(
            df_clean,
            term,
            date_start,
            date_end,
            measure='spearman_correlation',
            independent_threshold=0.33
        )
        networks.append(G)
        metadata.append({
            'term': term,
            'date_start': date_start,
            'date_end': date_end,
            'n_nodes': G.number_of_nodes(),
            'n_edges': G.number_of_edges()
        })
    
    layer_networks[term] = networks
    layer_metadata[term] = metadata

print(f"\nComputed intra-layer networks for {len(layer_networks)} terms")

In [ ]:
# Summarize layer statistics
print("Layer Statistics (per term):")
print("Term\t\tAvg Nodes\tAvg Edges\tDensity Range")
for term in terms_unique:
    if term in layer_metadata:
        metadata = layer_metadata[term]
        n_nodes_avg = np.mean([m['n_nodes'] for m in metadata])
        n_edges_avg = np.mean([m['n_edges'] for m in metadata])
        densities = []
        for m in metadata:
            if m['n_nodes'] > 1:
                d = m['n_edges'] / (m['n_nodes'] * (m['n_nodes'] - 1) / 2)
                densities.append(d)
        dens_range = f"[{np.min(densities):.3f}, {np.max(densities):.3f}]"
        print(f"{term}\t\t{n_nodes_avg:.1f}\t\t{n_edges_avg:.1f}\t\t{dens_range}")

## Chapter C: Multi-Layer Network Structure

Create inter-layer edges connecting the same source across different terms, forming a yield curve network.

In [ ]:
# Create a multi-layer network structure (multiplex graph)
# Node ID format: (source, term)
# Intra-layer edges: source correlations within a term
# Inter-layer edges: same source across terms (with weight 1.0, fully connected)

def build_multilayer_network(
    layer_graphs: dict,  # {term: G_t} where G_t is a NetworkX graph
    terms_list: list,
    inter_layer_weight: float = 1.0
) -> nx.Graph:
    """
    Build a multi-layer network from single-layer graphs.
    Nodes: (source, term) tuples
    Edges: intra-layer (correlation) + inter-layer (yield curve curve connectivity)
    """
    M = nx.Graph()
    
    # Add intra-layer edges (from each layer's network)
    for term in terms_list:
        if term not in layer_graphs:
            continue
        G_term = layer_graphs[term]
        for u, v, data in G_term.edges(data=True):
            # Node IDs: (source, term)
            node_u = (u, term)
            node_v = (v, term)
            weight = 1.0 - data.get('weight', 0.5)  # Convert distance to similarity
            M.add_edge(node_u, node_v, weight=weight, layer='intra', term=term)
    
    # Add inter-layer edges (same source across terms)
    # Collect all sources
    sources = set()
    for term in terms_list:
        if term in layer_graphs:
            sources.update(layer_graphs[term].nodes())
    
    # Connect each source across consecutive terms
    for source in sources:
        for i in range(len(terms_list) - 1):
            term1, term2 = terms_list[i], terms_list[i + 1]
            node1 = (source, term1)
            node2 = (source, term2)
            # Add inter-layer edge
            M.add_edge(node1, node2, weight=inter_layer_weight, layer='inter', source=source)
    
    return M

print("Function defined: build_multilayer_network")

In [ ]:
# Build multi-layer networks for the first time window
multilayer_networks = []
window_indices = []

for t_idx, (date_start, date_end) in enumerate(tqdm(windows, desc='Building multi-layer networks')):
    # Collect single-layer graphs for this time window
    layer_graphs_t = {}
    for term in terms_unique:
        if t_idx < len(layer_networks[term]):
            layer_graphs_t[term] = layer_networks[term][t_idx]
    
    # Build multi-layer network
    M_t = build_multilayer_network(
        layer_graphs_t,
        terms_unique,
        inter_layer_weight=1.0
    )
    
    multilayer_networks.append(M_t)
    window_indices.append((date_start, date_end))

print(f"\nBuilt {len(multilayer_networks)} multi-layer networks")

In [ ]:
# Analyze multi-layer network structure
print("Multi-Layer Network Statistics\n")
print("Window Index\tDate Start\tDate End\t\tNodes\tEdges\tLayers")
print("-" * 80)
for t_idx, (M, (date_start, date_end)) in enumerate(zip(multilayer_networks, window_indices)):
    n_nodes = M.number_of_nodes()
    n_edges = M.number_of_edges()
    
    # Count layers (unique terms present)
    terms_present = set()
    for node in M.nodes():
        if isinstance(node, tuple):
            terms_present.add(node[1])
    n_layers = len(terms_present)
    
    print(f"{t_idx}\t\t{date_start}\t{date_end}\t{n_nodes}\t{n_edges}\t{n_layers}")
    
    if t_idx > 5:
        print("...")
        break

## Chapter D: Multi-Layer Network Metrics

Compute centrality and community metrics accounting for multi-layer structure.

In [ ]:
# Compute per-layer and multi-layer centrality metrics
def compute_multilayer_metrics(M: nx.Graph, terms_list: list) -> dict:
    """
    Compute centrality metrics for multi-layer network.
    Returns: per-layer and aggregate metrics.
    """
    metrics = {
        'total_nodes': M.number_of_nodes(),
        'total_edges': M.number_of_edges(),
        'density': nx.density(M),
        'n_components': nx.number_connected_components(M),
        'per_layer': {}
    }
    
    # Per-layer metrics
    for term in terms_list:
        # Extract subgraph for this term
        nodes_in_layer = [n for n in M.nodes() if isinstance(n, tuple) and n[1] == term]
        if len(nodes_in_layer) > 0:
            G_layer = M.subgraph(nodes_in_layer)
            metrics['per_layer'][term] = {
                'n_nodes': len(nodes_in_layer),
                'n_edges': G_layer.number_of_edges(),
                'density': nx.density(G_layer) if len(nodes_in_layer) > 1 else 0,
                'n_components': nx.number_connected_components(G_layer)
            }
    
    return metrics

print("Function defined: compute_multilayer_metrics")

In [ ]:
# Compute metrics for all multi-layer networks
multilayer_metrics_list = []

for t_idx, M in enumerate(tqdm(multilayer_networks, desc='Computing metrics')):
    metrics = compute_multilayer_metrics(M, terms_unique)
    metrics['window_idx'] = t_idx
    metrics['date_start'] = window_indices[t_idx][0]
    metrics['date_end'] = window_indices[t_idx][1]
    multilayer_metrics_list.append(metrics)

print(f"Computed metrics for {len(multilayer_metrics_list)} windows")

# Convert to DataFrame for analysis
metrics_data = []
for m in multilayer_metrics_list:
    metrics_data.append({
        'window_idx': m['window_idx'],
        'date_start': m['date_start'],
        'date_end': m['date_end'],
        'total_nodes': m['total_nodes'],
        'total_edges': m['total_edges'],
        'density': m['density'],
        'n_components': m['n_components']
    })

metrics_df = pl.DataFrame(metrics_data)
print("\nMulti-layer network metrics:")
print(metrics_df.head(10))

In [ ]:
# metrics_df.sample(10).sort(['date_start'])
metrics_df.filter(pl.col("n_components")>1).sample(10).sort(['date_start'])

## Chapter E: Temporal Analysis of Multi-Layer Evolution

Analyze how the multi-layer network structure changes over time.

In [ ]:
# Visualize multi-layer network evolution (plotnine)
from plotnine import (
    ggplot,
    aes,
    geom_line,
    facet_wrap,
    labs,
    theme,
    theme_bw,
    scale_color_manual,
    guides,
    element_text,
    element_blank,
)

METRIC_SPECS = {
    "total_nodes": ("Network Size Growth", "#0284c7"),
    "total_edges": ("Edge Count Evolution", "#7c3aed"),
    "density": ("Connectivity Density", "#059669"),
    "n_components": ("Network Fragmentation", "#dc2626"),
}

plot_df = (
    metrics_df.select(["window_idx", *METRIC_SPECS.keys()])
    .to_pandas()
    .melt(id_vars="window_idx", var_name="metric", value_name="value")
)
plot_df["facet_title"] = plot_df["metric"].map(lambda m: METRIC_SPECS[m][0])
color_map = {m: spec[1] for m, spec in METRIC_SPECS.items()}

evolution_plot = (
    ggplot(plot_df, aes("window_idx", "value", color="metric", group="metric"))
    + geom_line(size=1.1, alpha=0.95)
    + facet_wrap("~ facet_title", ncol=2, scales="free_y")
    + scale_color_manual(values=color_map)
    + guides(color=None)
    + labs(
        title="Multi-Layer Network Evolution Over Time",
        x="Window Index",
        y="",
    )
    + theme_bw(base_size=13)
    + theme(
        figure_size=(14, 8),
        plot_title=element_text(size=16, weight="bold", ha="center"),
        strip_text=element_text(size=12, weight="bold"),
        strip_background=element_blank(),
        axis_text=element_text(size=11),
        axis_title_x=element_text(size=12, margin={"t": 10}),
        panel_grid_major=element_blank(),
        panel_grid_minor=element_blank(),
    )
)

evolution_plot.save(
    "multilayer_evolution.png",
    dpi=150,
    width=14,
    height=8,
    verbose=False,
)
evolution_plot


In [ ]:
# Per-layer statistics over time
print("Per-Layer Statistics Over Time\n")

per_layer_data = []
for m in multilayer_metrics_list:
    for term, layer_metrics in m.get('per_layer', {}).items():
        per_layer_data.append({
            'window_idx': m['window_idx'],
            'date_end': m['date_end'],
            'term': term,
            'n_nodes': layer_metrics['n_nodes'],
            'n_edges': layer_metrics['n_edges'],
            'density': layer_metrics['density'],
            'n_components': layer_metrics['n_components']
        })

per_layer_df = pl.DataFrame(per_layer_data)

print("Sample per-layer metrics (first 20 rows):")
print(per_layer_df.head(20))

print("\nAverage per-layer metrics:")
per_layer_summary = per_layer_df.group_by('term').agg([
    pl.col('n_nodes').mean().round(1).alias('avg_nodes'),
    pl.col('n_edges').mean().round(1).alias('avg_edges'),
    pl.col('density').mean().round(3).alias('avg_density'),
]).sort('term')
print(per_layer_summary)

### Interactive network visualization (pyvis)

Drag/zoom the graph below. Intra-layer edges are color-coded by weight; inter-layer edges are purple.


In [ ]:
import base64

from IPython.display import HTML, display

from tgraphportfolio.analysis.pyvis_plot import multilayer_graph_to_html

w0_start, w0_end = window_indices[0]
html = multilayer_graph_to_html(
    multilayer_networks[0],
    title=f"Multi-Layer Network (window 0: {w0_start} → {w0_end})",
    height="900px",
)
encoded = base64.b64encode(html.encode("utf-8")).decode("ascii")
display(
    HTML(
        "<p style='color:#cbd5e1;font-family:Segoe UI,sans-serif;margin:0 0 8px 0'>"
        "Grid: columns = issuers, rows = terms (short → long). "
        "Scroll to zoom, drag to pan, hover for node/edge details. "
        "Bottom-right buttons: zoom / fit."
        "</p>"
        f'<iframe src="data:text/html;base64,{encoded}" '
        'width="100%" height="900px" frameborder="0" '
        'style="border:1px solid #334155; background:#0f172a;"></iframe>'
    )
)


## Chapter F: Inter-Layer Connectivity Analysis

Analyze how yield curve structure (inter-layer connectivity) evolves over time.

In [ ]:
# Count inter-layer vs intra-layer edges
def count_edge_types(M: nx.Graph) -> dict:
    """
    Count edges by type (intra-layer vs inter-layer).
    """
    intra_edges = 0
    inter_edges = 0
    
    for u, v, data in M.edges(data=True):
        edge_layer = data.get('layer', 'unknown')
        if edge_layer == 'intra':
            intra_edges += 1
        elif edge_layer == 'inter':
            inter_edges += 1
    
    return {
        'intra_edges': intra_edges,
        'inter_edges': inter_edges,
        'total_edges': intra_edges + inter_edges
    }

edge_type_data = []
for t_idx, M in enumerate(multilayer_networks):
    counts = count_edge_types(M)
    counts['window_idx'] = t_idx
    counts['date_end'] = window_indices[t_idx][1]
    edge_type_data.append(counts)

edge_type_df = pl.DataFrame(edge_type_data)
edge_type_df = edge_type_df.with_columns([
    (100.0 * pl.col('intra_edges') / pl.col('total_edges')).alias('pct_intra'),
    (100.0 * pl.col('inter_edges') / pl.col('total_edges')).alias('pct_inter')
])

print("Edge Type Distribution Over Time:")
print(edge_type_df.head(15))

In [ ]:
# Visualize edge type evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Intra-Layer vs Inter-Layer Connectivity', fontsize=14, fontweight='bold')

# Plot 1: Absolute edge counts
ax = axes[0]
ax.plot(edge_type_df['window_idx'], edge_type_df['intra_edges'], marker='o', label='Intra-layer', linewidth=2, color='#0ea5e9')
ax.plot(edge_type_df['window_idx'], edge_type_df['inter_edges'], marker='s', label='Inter-layer', linewidth=2, color='#a78bfa')
ax.set_xlabel('Window Index')
ax.set_ylabel('Edge Count')
ax.set_title('Edge Counts by Type')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#0f172a')

# Plot 2: Percentage stacked
ax = axes[1]
ax.fill_between(edge_type_df['window_idx'], 0, edge_type_df['pct_intra'], alpha=0.7, label='Intra-layer', color='#0ea5e9')
ax.fill_between(edge_type_df['window_idx'], edge_type_df['pct_intra'], 100, alpha=0.7, label='Inter-layer', color='#a78bfa')
ax.set_xlabel('Window Index')
ax.set_ylabel('Percentage (%)')
ax.set_title('Edge Type Composition')
ax.set_ylim([0, 100])
ax.legend(loc='center left')
ax.grid(True, alpha=0.3, axis='y')
ax.set_facecolor('#0f172a')

for ax in axes.flat:
    ax.tick_params(colors='#cbd5e1')
    ax.spines['bottom'].set_color('#475569')
    ax.spines['left'].set_color('#475569')
    ax.xaxis.label.set_color('#cbd5e1')
    ax.yaxis.label.set_color('#cbd5e1')
    ax.title.set_color('#cbd5e1')
    for spine in ax.spines.values():
        spine.set_visible(True)

plt.tight_layout()
plt.savefig('multilayer_edge_types.png', dpi=150, facecolor='#0f172a', edgecolor='none')
plt.show()

print("Edge type evolution plot saved")

## Chapter G: Community Detection in Multi-Layer Networks

Detect communities within and across layers to identify groups of correlated issuers.

In [ ]:
# Community detection on multi-layer networks
# Use modularity-based approach treating multi-layer as single network
from graspologic.embed import AdjacencySpectralEmbed
from sklearn.cluster import KMeans

def detect_multilayer_communities(M: nx.Graph, n_components: int = 2) -> dict:
    """
    Detect communities using spectral embedding + KMeans on multi-layer network.
    """
    if M.number_of_nodes() < 3:
        return {'communities': {}, 'n_clusters': 0}
    
    # Convert to adjacency matrix
    A = nx.to_numpy_array(M, nodelist=sorted(M.nodes()))
    node_list = sorted(M.nodes())
    
    try:
        # Spectral embedding
        ase = AdjacencySpectralEmbed(
            n_components=min(n_components, len(node_list) - 1),
            check_lcc=False,
        )
        X = ase.fit_transform(A)
        if isinstance(X, tuple):
            X = np.hstack(X)
        
        # KMeans clustering
        kmeans = KMeans(n_clusters=min(3, len(node_list)), random_state=0, n_init=10)
        labels = kmeans.fit_predict(X)
        
        communities = {node: int(label) for node, label in zip(node_list, labels)}
        
        return {
            'communities': communities,
            'n_clusters': len(set(labels)),
            'inertia': float(kmeans.inertia_)
        }
    except Exception as e:
        return {'communities': {}, 'n_clusters': 0, 'error': str(e)}

print("Function defined: detect_multilayer_communities")

In [ ]:
len(multilayer_networks)

In [ ]:
# Detect communities for first few time windows
multilayer_communities = []

for t_idx, M in enumerate(tqdm(multilayer_networks, desc='Detecting communities')):
    result = detect_multilayer_communities(M, n_components=3)
    result['window_idx'] = t_idx
    result['date_end'] = window_indices[t_idx][1]
    multilayer_communities.append(result)

print(f"\nCommunity detection results:")
for i, result in enumerate(multilayer_communities[:5]):
    print(f"Window {i}: {result['n_clusters']} communities, inertia={result.get('inertia', 'N/A')}")

## Future Development & Exploration Suggestions

This notebook provides a foundation for multi-layer yield curve network analysis. The following sections outline promising directions for future work.

### A. Enhanced Inter-Layer Connectivity

**Current approach**: Simple direct connections between same source across terms.

**Future directions**:
- **Weighted inter-layer edges**: Use term structure fitting (splines, factor models) to weight connections by yield curve curvature or slope
- **Term-specific connectivity**: Different inter-layer weights for adjacent terms vs. distant terms (6MÔåÆ1Y closer than 6MÔåÆ10Y)
- **Coupling coefficients**: Learn optimal inter-layer weights from historical co-movement between terms
- **Multi-factor inter-layer**: Connect sources not just directly but through shared factors (credit risk, duration, sector)

**Implementation patterns** (from honml):
- Multi-relational graphs with typed edges (correlation vs. spread vs. curve)
- Tensor decomposition for factor-based inter-layer weights
- Coupled network learning to optimize inter-layer structure

### B. Temporal Dynamics & Network Evolution

**Current approach**: Independent snapshots per rolling window.

**Future directions**:
- **Node persistence tracking**: Follow individual issuers' centrality trajectories across windows (similar to dax_network_evolution.ipynb Chapter D)
- **Edge birth/death processes**: Identify when correlations stabilize vs. break (market regime changes)
- **Quadratic variation of centrality**: Measure how quickly network hubs change (stress regime detector)
- **Markov chain analysis**: Model network state transitions (from tight correlation to crisis fragmentation)

**Visualization**: Extended centrality heatmaps showing source ├ù term ├ù time 3D evolution

### C. Multi-Layer Community Detection

**Current approach**: Spectral embedding treating all layers as single network.

**Future directions**:
- **Layer-aware modularity**: Optimize modularity with explicit inter/intra-layer weights (Br├│dka et al., Battiston et al.)
- **Overlapping communities**: Allow sources to belong to multiple communities across different maturity segments
- **Coherence measures**: Quantify which sources maintain consistent community membership across terms
- **Community evolution**: Track community lifecycle (emergence, growth, dissolution, merger) across windows
- **Multilevel algorithms**: Apply Louvain-style optimization accounting for layer structure

**References**: Look for implementations in:
- `muxViz` (Python library for multilayer network visualization)
- `pymnet` (Python MUltilayer NETwork toolkit)
- Recent papers on inter-dependent networks and multi-layer modularity

### D. Centrality & Influence Metrics

**Current approach**: Basic connectivity statistics.

**Future directions**:
- **Tensor-based centrality**: Extend eigenvector/PageRank centrality to tensors (multi-layer adjacency)
- **Betweenness accounting for layers**: Path-based centrality that measures how critical each issuer is across term structures
- **Spreading dynamics**: Simulate information/shock propagation through multi-layer network (epidemic models)
- **Cross-layer influence**: Measure how changes in 5Y term affect 2Y and 10Y behavior through network (impulse response)

**Application**: Identify systemically important issuers in yield curve ecosystem

### E. Integration with Existing TGraph Pipeline

**Current status**: Notebook-only analysis.

**Future directions**:
- **GUI extension**: Add multi-layer tab to TGraph GUI similar to evolution_tab but showing:
  - Layer selector (dropdown for term)
  - Multi-layer view toggle (show all layers + inter-layer edges simultaneously)
  - Per-layer network visualization with inter-layer connectors highlighted
- **Configuration options**: 
  - Inter-layer weight formula
  - Term grouping (all terms vs. selected maturities)
  - Community detection algorithm selection
- **Export capabilities**: Save multi-layer networks to formats (GraphML, JSON) preserving layer information
- **Batch analysis**: Compute statistics for all windows automatically

### F. Stress Testing & Correlation Breakdown

**Current approach**: Single correlation measure over rolling window.

**Future directions**:
- **Conditional correlation breakdown detection**: Identify windows where normal correlations collapse (market stress indicator)
- **Layer-specific stress**: Detect if certain maturity segments become disconnected (liquidity dry-ups)
- **Tail dependence analysis**: Use Clayton copula or conditional correlation on extreme returns (Chapter B in notebook)
- **Yield spread networks**: Separate network for bid-ask spreads (liquidity view) from rates (fundamental view)
- **Crisis versus normal regime**: Segment analysis showing different network topology in stress periods

**Application**: Early warning system for market dislocations

### G. Yield Curve Factors & Network Decomposition

**Current approach**: Direct rate correlations.

**Future directions**:
- **Factor residual networks**: Build networks from PCA residuals after removing level/slope/curvature (issuer-specific correlation)
- **Nelson-Siegel decomposition**: Use fitted factors as edge weights or community markers
- **Multi-factor inter-layer**: Separate network layers for each factor (level effects, slope effects, etc.)
- **Arbitrage-free models**: Incorporate yield curve constraints (no-arbitrage conditions) into network construction

**Data structure**:
```python
# Enhanced network nodes could carry:
node_attrs = {
    'source': 'ACME Inc',
    'term': 'Y005p0',  # 5Y maturity
    'yield': 3.25,
    'duration': 4.8,
    'credit_rating': 'A',
    'sector': 'Financials',
    'level_factor': 0.42,
    'slope_factor': -0.15,
    'curve_factor': 0.08
}
```

### H. Visualization & Interaction

**Current approach**: Static plots (matplotlib).

**Future directions**:
- **Interactive 3D visualization**: x=term, y=source, z=time showing evolving multi-layer structure (plotly/vispy)
- **Animated network evolution**: Visualize layer formation/dissolution and community merging over time (with frame-by-frame playback)
- **Alluvial diagrams**: Show how issuer communities evolve across terms and time
- **Correlation heatmaps with hierarchy**: Dendrogram clustering of issuers by their cross-term behavior
- **Accessibility**: Export to D3.js or Cytoscape.js for web-based interactive exploration

**Tools to explore**:
- `PyVis`: Already used in TGraph; extend for multi-layer views
- `Plotly 3D`: Network visualization in 3D space
- `Vispy`: GPU-accelerated visualization for large networks
- `graphviz`/`Sugiyama`: Hierarchical layout for term ├ù term structure

### I. Statistical Inference & Hypothesis Testing

**Current approach**: Descriptive statistics only.

**Future directions**:
- **Network hypothesis tests**: Test if observed multi-layer structure differs significantly from null model
- **Latent position model**: Fit multi-layer latent position models (omnibus embedding across layers)
- **Isomorphism detection**: Test if term-specific networks are significantly similar (correlation stability)
- **Change point detection**: Identify when network topology fundamentally shifts (using graspologic's latent position tests)

**Implementation**: Use graspologic's inference module extended to multi-layer case

### J. Integration with honml Patterns

Patterns extracted from honml repository:

1. **Signal Subgraph Detection** (Ch8)
   - Identify subgraph of edges that best discriminate between market regimes
   - Use graspologic.subgraph.SignalSubgraph on multi-layer networks

2. **Statistical Testing on Graphs**
   - Fisher's exact test for edge presence across issuers
   - Latent position tests to detect significant network changes

3. **Classification Using Graph Features**
   - Train classifier on multi-layer network features to predict future regime (stress vs normal)
   - Use Bernoulli Naive Bayes with signal subgraph edges

4. **Embedding & Representation**
   - Omnibus embedding for multi-layer comparison
   - Joint embedding of sources across all terms simultaneously

### K. Domain-Specific Applications

**Credit risk management**:
- Multi-layer network as portfolio risk model (term-specific concentrations)
- Issuer centrality as proxy for default clustering risk
- Community structure as proxy for sector/geopolitical concentration

**Asset pricing**:
- Network-based factors for yield spread models
- Leverage centrality or betweenness as predictors for future spread changes
- Use multi-layer embedding as input to term structure models

**Market microstructure**:
- Intra-day multi-layer networks (changes within a trading day)
- Liquidity provision network (bid-ask spread correlations)
- Trading flow networks (who trades with whom across terms)

**Regulatory monitoring**:
- Systemic risk indicators from multi-layer network metrics
- Contagion risk: how quickly does stress in one term spread to others?
- Concentration risk: which issuers are too central to fail?

## Summary of Immediate Next Steps

1. **Validate against real yield curve structure**: Ensure inter-layer weights reflect actual yield curve mechanics
2. **Scale analysis**: Run full analysis on complete date range and all terms
3. **Add factor models**: Integrate PCA/Nelson-Siegel decomposition to isolate issuer-specific effects
4. **Community validation**: Compare detected communities with known credit sectors and rating groups
5. **Temporal analysis**: Extend to full window range; identify regime changes and community lifecycle
6. **GUI integration**: Design mockups for multi-layer network visualization in TGraph application
7. **Publication-ready metrics**: Formalize multi-layer modularity, layer-aware betweenness, tensor centrality